## Mesh Processing

#### Objective
- We'll now look into the remaining 3 mesh operations that can change either
    1. Mesh Topology of Object
    2. Geometric Structure of Object
    3. Both

Changing the topological geometry of the mesh means changing the geometric properties defining the mesh 
    - Face (Triangle)
    - Edge (Two Points)
    - Vertices (Points)

<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.25.22.png" height="325" width="325"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.26.07.png" height="325" width="325"/>


Notice we can do without changing the mesh just yet:

#### Example: Simple Movements of an Object
- Using Rigid Body transformations
    1. Rotations of the whole object.
    2. Tranlation of the whole object.
- The positions of the vertices of the mesh changed, but the relative positions between each mesh entity remained the same.
- Indeed the object changed position be the object itself remains the same. 


<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.31.57.png" height="325" width="325"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.32.29.png" height="325" width="325"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.32.48.png" height="325" width="325"/>

##### Q. How can this be done?
##### Multiply the vertices by a single matrix representing the motion before drawing them!
- So in the GPU rendering, this can be done in the modelling transformation step, since this positions the object in the 3D world. 

#### Extension to Affine Transformations:
- We can also modify the shape of the object and change the position of the vertices, but we still don't need to go down to the resolution of the vertices to obtain this result
- We can apply matrix multiplication is enough to create Affine motions (Think Shearing).

#### Q. So when do we need to change the positions of vertices?

#### Moving parts of the object

- In order to change specific parts of composed 3D object we may need to change the position of those desired vertices in order to create independent movement.
- Important to note that this won't change the topology of the object as a whole. Consider the dragon, it's still moving it's head and wings etc, but fortunately we still recognise that it's a dragon. 

<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.40.59.png" height="325" width="325"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.41.15.png" height="325" width="325"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.41.34.png" height="325" width="325"/>

- Suppose we tried to scan a real object to produce the 3D Virtual Object (Like the Micheal Angelo Project). We could obtain errors in the representation due to the physical limitations of the device/computation. 
    - Outlier points could arise from the triangulation
    - Noise, which change the positons of points close to the real surface, by small pertubation, so the surface will appear bumpy and not smooth.



<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.46.05.png" height="325" width="325"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.48.19.png" height="325" width="325"/>


#### Solving Noise By Average

- $\text{Laplacian Smoothing}$ Replace the position of a vertex, by the average position of its neighbors. $$v_{new} = \frac{1}{n}\sum_{i=1}^{n}v_i$$
- To find the neighbors of a vertex efficiently we need the V-V adjacency relation.

<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.49.59.png" height="325" width="325"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.50.21.png" height="325" width="325"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.51.57.png" height="325" width="325"/>

- If we apply $\text{Laplacian Smoothing}$ many times the mesh will eventually shrink to a single point, as we're always averaging over and over again.

<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.54.30.png" height="250" width="250"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.54.44.png" height="250" width="250"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.54.58.png" height="250" width="250"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.55.14.png" height="250" width="250"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 11.55.33.png" height="250" width="250"/>


- We could solve this by only moving the result of the average to part of the positional direction of the new calculated point. 
$$ Average = \frac{1}{n}\sum_{i=1}^{n}v_i$$
$$ \vec{offset} = \text{Average} - v_{old}$$
$$ v_{new} = v_{old} - \lambda\vec{offset}$$
- Notice we obtained a parametric equation of a line (specifically Line segment constraint).
- If we change $\lambda < 0$ the point will moving moving up, we end up sharpening the image.

<img src="image_U9/9.8/Screenshot 2025-06-03 at 12.01.10.png" height="325" width="325"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 12.01.48.png" height="325" width="325"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 12.02.03.png" height="325" width="325"/>

- Again $\text{Laplacian Smoothing}$ and $\text{Laplacian Shaerpening}$ DON'T CHANCE MESH TOPOLOGY but only OBJECT GEOMETRY.

#### How do we change topology with Geometry?
- This would mean changing the triangulation without chagning the vertex positions, So for a given set of points in 2D, we saw that there many possible triangulations, but not every set of triangles (defined over the set of points) provide a valid triangulation. The same applies with 3D mesh.


<img src="image_U9/9.8/Screenshot 2025-06-03 at 12.05.16.png" height="325" width="325"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 12.07.16.png" height="325" width="325"/>

##### Reminder on 3D Mesh Constraint:
1. Orientable 
2. 2-Manifold
3. Watertight Surface


- Consider a pair of neighboring Triangles, when we create a convex quadilateral we can easiler swap the connecting edge and create two different triangles using the same set of (4) vertices . If the Pair of triangles made a non-convex shape, the swapping will create an INVALID triangulation.

<img src="image_U9/9.8/Screenshot 2025-06-03 at 12.08.50.png" height="325" width="325"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 12.09.46.png" height="325" width="325"/>

- Having said that In 3D although possible it's difficult to change the topolgy without changing the geometry, consdier the following simple example to understand:
The 3D Convex Shape is the space, by the triangles forming it, are different so changing the mesh will necessarily change the geometry.

<img src="image_U9/9.8/Screenshot 2025-06-03 at 12.11.20.png" height="325" width="325"/>
<img src="image_U9/9.8/Screenshot 2025-06-03 at 12.11.42.png" height="325" width="325"/>

$\text{Remeshing = Changing the mesh topolgy (trying to avoid chaing the geometry too much)}$

# Summary: Geometry vs Topology in 3D Operations

| **Technique**               | **Aims to Change**       | **Geometry or Topology?** | **Description**                                                                 | **Example Image**               |
|----------------------------|--------------------------|----------------------------|----------------------------------------------------------------------------------|---------------------------------|
| **Animation**              | Vertex positions over time | **Geometry**               | Moves vertices (e.g., skeletal or morph animation) but keeps mesh topology fixed. | <img src="image_U9/9.8/Screenshot 2025-06-03 at 11.41.34.png" height="120" width="120"/> |
| **Mesh Editing**           | Shape and structure       | **Usually Geometry**, sometimes Topology | Manual or scripted modifications like moving vertices, extruding, or cutting holes. | <img src="image_U9/9.8/Screenshot 2025-06-03 at 11.26.07.png" height="120" width="120"/> |
| **Mesh Smoothing / Sharpening** | Surface details (smoothness or edge sharpness) | **Geometry**               | Alters vertex positions to reduce noise (smoothing) or enhance edges (sharpening). | <img src="image_U9/9.8/Screenshot 2025-06-03 at 11.54.30.png" height="120" width="120"/> |
| **Re-meshing**             | Triangle distribution / density / regularity | **Topology** (and slightly Geometry) | Changes the mesh connectivity and face layout to improve uniformity or performance. | <img src="image_U9/9.3/Screenshot 2025-06-01 at 8.28.44.png" height="120" width="120"/> |




